In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

In [3]:
from clean import DataCleaner
import preprocess

In [4]:
df = pd.read_csv('data/movies.csv')

In [5]:
cleaner = DataCleaner(df)
df = cleaner.clean()

In [6]:
df = preprocess.split_date_to_year_and_month(df)

In [7]:
features = ['year', 'month', 'genre', 'orig_lang', 'revenue', 'budget_x', 'country']
target = ['score']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[target], test_size=0.20, random_state=42
)

In [9]:
# Genre: one column per genre, allowing multiple 1s per movie
def genre_lists(series):
    return series.fillna("").apply(
        lambda s: [g.strip() for g in s.split(",") if g.strip()]
    )

mlb = MultiLabelBinarizer()
train_genres = mlb.fit_transform(genre_lists(X_train["genre"]))
test_genres = mlb.transform(genre_lists(X_test["genre"]))

# Language: one column per language category
language_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
train_languages = language_encoder.fit_transform(
    X_train[["orig_lang"]].apply(lambda s: s.str.strip())
)
test_languages = language_encoder.transform(
    X_test[["orig_lang"]].apply(lambda s: s.str.strip())
)

# Country: one column per country category
country_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
train_country = country_encoder.fit_transform(
    X_train[["country"]].apply(lambda s: s.str.strip())
)
test_country = country_encoder.transform(
    X_test[["country"]].apply(lambda s: s.str.strip())
)

In [10]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ["year", "month", "budget_x", "revenue"]

scaler = StandardScaler()
train_numeric = scaler.fit_transform(X_train[numeric_cols])
test_numeric = scaler.transform(X_test[numeric_cols])

X_train_encoded = np.column_stack([
    train_numeric,
    train_genres,
    train_languages,
    train_country
])

X_test_encoded = np.column_stack([
    test_numeric,
    test_genres,
    test_languages,
    test_country
])

In [11]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_encoded, np.ravel(y_train))

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](121,)","[-0.83, 0.24,-1.88,..., 2.11,-7.01,-3.37]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,62.33
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,121
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(116)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](121,)","[120.41, 89.18, 83.97,..., 0. , 0. , 0. ]"


In [12]:
predictions = model.predict(X_test_encoded)

print(f"MSE: {np.sqrt(mean_squared_error(y_test, predictions)):.2f}")
print(f"R²:   {r2_score(y_test, predictions):.3f}")

MSE: 8.30
R²:   0.203


In [13]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)  # Higher alpha = stronger regularization
model.fit(X_train_encoded, np.ravel(y_train))

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [14]:
predictions = model.predict(X_test_encoded)

print(f"MSE: {np.sqrt(mean_squared_error(y_test, predictions)):.2f}")
print(f"R²:   {r2_score(y_test, predictions):.3f}")

MSE: 8.29
R²:   0.205


In [15]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1)
model.fit(X_train_encoded, np.ravel(y_train))

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [16]:
predictions = model.predict(X_test_encoded)

print(f"MSE: {np.sqrt(mean_squared_error(y_test, predictions)):.2f}")
print(f"R²:   {r2_score(y_test, predictions):.3f}")

MSE: 7.14
R²:   0.411


In [17]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=2000,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42
)

model.fit(X_train_encoded, np.ravel(y_train))

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",2000
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",15
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.0
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [18]:
predictions = model.predict(X_test_encoded)

print(f"MSE: {np.sqrt(mean_squared_error(y_test, predictions)):.2f}")
print(f"R²:   {r2_score(y_test, predictions):.3f}")

MSE: 6.84
R²:   0.459


In [19]:
errors = np.abs(np.ravel(y_test) - predictions)

print(f"Within 5 rating points:  {(errors <= 5).mean():.1%}")
print(f"Within 10 rating points: {(errors <= 10).mean():.1%}")

Within 5 rating points:  57.9%
Within 10 rating points: 86.3%


### Predictions for three unseen 2024 movies
Inputs collected on 2026-10-08 from IMDb indexed listings, with The Wild Robot worldwide revenue from Box Office Mojo and its budget from The Numbers. US theatrical release dates and country US are used; original language is English. Country here represents the selected release market, not all production countries. IMDb genres are mapped to the training vocabulary (Sci-Fi -> Science Fiction); Sport is omitted for Inside Out 2 because it is absent from the training vocabulary. Subgenre labels are omitted.

Sources: [Dune: Part Two](https://www.imdb.com/title/tt15239678/?lang=en), [Inside Out 2](https://www.imdb.com/title/tt22022452/?newac=true), [The Wild Robot IMDb](https://www.imdb.com/title/tt29623480/), [The Wild Robot box office](https://www.boxofficemojo.com/title/tt29623480/), [The Wild Robot budget](https://www.the-numbers.com/movie/Wild-Robot-The-(2024)).

These are inference examples outside the dataset. IMDb ratings are separate reference values, not verified ground truth for this dataset's score. No accuracy metric is computed from them. The model includes realized revenue, so these are post-release predictions.


In [20]:
# Load the collected inputs; run the training cells above first
new_movies = pd.read_csv("results/unseen_movie_inputs.csv")

new_movies_encoded = np.column_stack([
    scaler.transform(new_movies[numeric_cols]),
    mlb.transform(genre_lists(new_movies["genre"])),
    language_encoder.transform(new_movies[["orig_lang"]]),
    country_encoder.transform(new_movies[["country"]])
])

# model is the most recently fitted model (gradient boosting after Run All)
unseen_predictions = new_movies[["names", "imdb_rating_reference"]].copy()
unseen_predictions["predicted_score"] = model.predict(new_movies_encoded)
print(f"Model: {type(model).__name__}")
display(unseen_predictions.round(2))


Model: HistGradientBoostingRegressor


,names,imdb_rating_reference,predicted_score
0,Dune: Part Two,8.4,68.84
1,Inside Out 2,7.5,80.53
2,The Wild Robot,8.1,65.72
